PDI / PPI / LLM-only comparison across all 8 LLMs on the offensiveness dataset.

For each model, prints one screenshot-style table per dimension (Age, Gender, Occupation):
- rows: zero shot / few shot / persona / PPI / PDI
- columns: Avg + each subgroup, each with `Cov.` (coverage %) and `Δ` (mean |estimate − true theta| in percentage points)

Also prints mean human-sample allocation per subgroup (across trials).

Same hyperparameters as `evaluation_offensiveness_gpt52.ipynb`. Original PDI files not modified.

In [ ]:
import sys, os
# `utils.py` (file) shadows `utils/` (dir) at the repo root, so `from utils.inference ...`
# fails. Load `utils/inference.py` directly by adding the dir to sys.path.
_utils_dir = os.path.join(os.getcwd(), 'utils')
if _utils_dir not in sys.path:
    sys.path.insert(0, _utils_dir)

import numpy as np
import pandas as pd
from scipy.stats import norm as scipy_norm
from tqdm.notebook import tqdm
%load_ext autoreload
%autoreload 2

from inference import (
    train_sampling_rule,
    sampling_rule_predict,
    perspective_driven_inference,
)
from ppi_py import ppi_mean_pointestimate, ppi_mean_ci

In [ ]:
alpha        = 0.05
burnin_steps = 100
n_batches    = 3
n_human      = 200
random_state = 42
n_trials     = 20
tau          = 0.1
q_power      = 1.0

offensive_threshold = 2

demo_cols = ['gender', 'race', 'age', 'education']

# Subgroups grouped by dimension.
dimensions = {
    'Age':        ['Age 18-34', 'Age 35-49', 'Age 50+'],
    'Gender':     ['Woman', 'Man'],
    'Occupation': ['Employed', 'Unemployed', 'Self-employed', 'Student'],
}

group_mask_fns = {
    'Age 18-34':     lambda df: df['age'].isin(['18-24', '25-29', '30-34']),
    'Age 35-49':     lambda df: df['age'].isin(['35-39', '40-44', '45-49']),
    'Age 50+':       lambda df: df['age'].isin(['50-54', '54-59', '60-64', '>65']),
    'Woman':         lambda df: df['gender'] == 'Woman',
    'Man':           lambda df: df['gender'] == 'Man',
    'Employed':      lambda df: df['occupation'] == 'Employed',
    'Unemployed':    lambda df: df['occupation'] == 'Unemployed',
    'Self-employed': lambda df: df['occupation'] == 'Self-employed',
    'Student':       lambda df: df['occupation'] == 'Student',
}

all_groups = [g for names in dimensions.values() for g in names]

def mean_estimator(y, weights):
    y, weights = y[~np.isnan(y)], weights[~np.isnan(y)]
    return np.sum(y * weights) / np.sum(weights)

In [ ]:
all_models = [
    'gpt-5.2',
    'claude-sonnet-4.6',
    'claude-opus-4.6',
    'gemini-3.1-pro',
    'claude-haiku-4.5',
    'llama-3-8b',
    'mistral-large-2512',
    'gpt-oss-120b',
]

model_cols = {
    m: {
        'zs': f'{m} (zero shot prompting)',
        'fs': f'{m} (few shot)',
        'pp': f'{m} (persona prompt)',
    }
    for m in all_models
}

required_cols = [c for m in all_models for c in model_cols[m].values()]
data_mm = (
    pd.read_csv('data/raw_data_llm_offensiveness.csv')
      .query("gender != 'Non-binary'")
      .dropna(subset=required_cols)
      .sample(frac=1, random_state=random_state)
      .reset_index(drop=True)
      .copy()
)
data_mm['human_true'] = (data_mm['offensiveness'] >= offensive_threshold).astype(float)
N_mm = len(data_mm)

demo_features_mm = pd.get_dummies(data_mm[demo_cols]).astype(float).values
group_masks  = [group_mask_fns[g](data_mm).values for g in all_groups]
true_thetas  = np.array([data_mm.loc[m, 'human_true'].mean() for m in group_masks])
group_idx    = {g: i for i, g in enumerate(all_groups)}

print(f'N = {N_mm}')
for g, mask, theta in zip(all_groups, group_masks, true_thetas):
    print(f'  {g:<14} size={int(mask.sum()):>4d}  true theta={theta:.3f}')

In [ ]:
METHODS = ['zero shot', 'few shot', 'persona', 'PPI', 'PDI']

def run_trials(Hhat_zs, Hhat_fs, Hhat_pp, Hhat_best,
               data, demo_features, group_masks, true_thetas):
    """Run all 5 methods across `n_trials`. Returns per-method (n_trials, n_groups) arrays."""
    n_groups = len(group_masks)
    N_loc    = len(data)

    ests          = {m: np.full((n_trials, n_groups), np.nan) for m in METHODS}
    covered       = {m: np.full((n_trials, n_groups), np.nan) for m in METHODS}
    samples_per_g = np.zeros((n_trials, n_groups), dtype=int)

    batch_size_loc = (N_loc - burnin_steps) // n_batches
    z_crit = scipy_norm.ppf(1 - alpha / 2)

    def compute_q_loc(rule, features):
        return np.clip(sampling_rule_predict(rule, features), 1e-8, None) ** q_power

    for trial in range(n_trials):
        np.random.seed(random_state + trial * 13)

        # PPI: uniform random sample of size n_human used both as labeled set and to define unlabeled set.
        ppi_idx = np.random.choice(N_loc, size=n_human, replace=False)
        ppi_selected = np.zeros(N_loc, dtype=bool)
        ppi_selected[ppi_idx] = True

        # PDI: adaptive sampling.
        H  = np.full(N_loc, np.nan)
        SP = np.zeros(N_loc)
        SD = np.zeros(N_loc)

        H[:burnin_steps]  = data['human_true'].values[:burnin_steps]
        SP[:burnin_steps] = 1.0
        SD[:burnin_steps] = 1.0

        sampling_rule = train_sampling_rule(
            demo_features[:burnin_steps],
            (H[:burnin_steps] - Hhat_best[:burnin_steps]) ** 2,
            seed=random_state)
        q = compute_q_loc(sampling_rule, demo_features)

        for b in range(n_batches):
            batch_inds = (
                np.arange(burnin_steps + b * batch_size_loc,
                          burnin_steps + (b + 1) * batch_size_loc)
                if b < n_batches - 1
                else np.arange(burnin_steps + b * batch_size_loc, N_loc)
            )
            remaining = n_human - int(SD.sum())
            if remaining <= 0:
                break
            target = min(round(remaining / (n_batches - b)), remaining, len(batch_inds))

            p = q[batch_inds]; p = p / p.sum()
            p = (1 - tau) * p + tau / len(batch_inds); p = p / p.sum()

            sel = np.random.choice(len(batch_inds), size=target, replace=False, p=p)

            H[batch_inds[sel]] = data['human_true'].values[batch_inds[sel]]
            SD[batch_inds] = 0.0; SD[batch_inds[sel]] = 1.0
            SP[batch_inds] = np.clip(target * p, 1e-4, 1.0)

            if b < n_batches - 1:
                lab = np.where(~np.isnan(H))[0]
                sampling_rule = train_sampling_rule(
                    demo_features[lab], (H[lab] - Hhat_best[lab]) ** 2,
                    seed=random_state)
                q = compute_q_loc(sampling_rule, demo_features)

        for g, (mask, true_theta) in enumerate(zip(group_masks, true_thetas)):
            samples_per_g[trial, g] = int(SD[mask].sum())

            llm_mask    = mask & ppi_selected
            g_sampled   = mask & ppi_selected
            g_unsampled = mask & ~ppi_selected

            # LLM-only baselines on the PPI-labeled subset (matches original notebook)
            for lm_key, Hhat_lm in [('zero shot', Hhat_zs),
                                     ('few shot',  Hhat_fs),
                                     ('persona',   Hhat_pp)]:
                if llm_mask.sum() >= 2:
                    lm_est = Hhat_lm[llm_mask].mean()
                    se2 = max(lm_est * (1 - lm_est), 1e-6) / llm_mask.sum()
                    se  = np.sqrt(se2)
                    ests[lm_key][trial, g]    = lm_est
                    covered[lm_key][trial, g] = int(lm_est - z_crit * se <= true_theta <= lm_est + z_crit * se)

            # PPI++
            if g_sampled.sum() >= 2 and g_unsampled.sum() >= 1:
                Y_lab      = data['human_true'].values[g_sampled]
                Yhat_lab   = Hhat_best[g_sampled]
                Yhat_unlab = Hhat_best[g_unsampled]
                est    = ppi_mean_pointestimate(Y_lab, Yhat_lab, Yhat_unlab, lam=None)
                lb, ub = ppi_mean_ci(Y_lab, Yhat_lab, Yhat_unlab, alpha=alpha, lam=None)
                ests['PPI'][trial, g]    = est
                covered['PPI'][trial, g] = int(lb <= true_theta <= ub)

            # PDI
            if SD[mask].sum() >= 2:
                est, (lb, ub), _ = perspective_driven_inference(
                    mean_estimator,
                    Y=H[mask], Yhat=Hhat_best[mask],
                    sampling_probs=SP[mask], sampling_decisions=SD[mask],
                    alpha=alpha, lam=None,
                    n_resamples=100, n_resamples_lam=20)
                ests['PDI'][trial, g]    = est
                covered['PDI'][trial, g] = int(lb <= true_theta <= ub)

    return ests, covered, samples_per_g

In [ ]:
results = {}
human = data_mm['human_true'].values

for model in tqdm(all_models, desc='Models'):
    cols = model_cols[model]
    Hhat_zs = (data_mm[cols['zs']] >= offensive_threshold).astype(float).to_numpy()
    Hhat_fs = (data_mm[cols['fs']] >= offensive_threshold).astype(float).to_numpy()
    Hhat_pp = (data_mm[cols['pp']] >= offensive_threshold).astype(float).to_numpy()

    accs = {
        'zero shot': (Hhat_zs == human).mean(),
        'few shot':  (Hhat_fs == human).mean(),
        'persona':   (Hhat_pp == human).mean(),
    }
    best_variant = max(accs, key=lambda v: accs[v])
    Hhat_best = {'zero shot': Hhat_zs, 'few shot': Hhat_fs, 'persona': Hhat_pp}[best_variant]

    ests, covered, samples_per_g = run_trials(
        Hhat_zs, Hhat_fs, Hhat_pp, Hhat_best,
        data_mm, demo_features_mm, group_masks, true_thetas)

    results[model] = {
        'best_variant':  best_variant,
        'ests':          ests,
        'covered':       covered,
        'samples_per_g': samples_per_g,
    }

print('\nDone.')

In [ ]:
ci_level = int((1 - alpha) * 100)

def coverage_pct(arr_g):  # mean over trials, in %
    return np.nanmean(arr_g) * 100 if not np.all(np.isnan(arr_g)) else np.nan

def delta_pp(ests_g, true_theta):  # mean |est - theta| in percentage points
    diffs = np.abs(ests_g - true_theta)
    return np.nanmean(diffs) * 100 if not np.all(np.isnan(diffs)) else np.nan

def fmt(v, w, prec):
    return f'{"nan":>{w}}' if np.isnan(v) else f'{v:>{w}.{prec}f}'

def print_dim_table(model, dim, subgroups, ests_d, cov_d, true_thetas):
    cols = ['Avg'] + subgroups
    sub_idx = [group_idx[s] for s in subgroups]

    label_w = 26
    cell_w  = 7  # width per Cov / Delta cell
    sep_w   = 1  # space between Cov and Delta
    block_w = cell_w * 2 + sep_w  # width of one (Cov, Δ) pair

    # Top header: column groups
    top = ' ' * label_w
    for c in cols:
        top += '  ' + f'{c:^{block_w}}'
    print(top)

    sub = f'{"Method":<{label_w}}'
    for _ in cols:
        sub += '  ' + f'{"Cov.":>{cell_w}}' + ' ' + f'{"Δ":>{cell_w}}'
    print(sub)
    print('-' * len(sub))

    bv = results[model]['best_variant']
    method_labels = {
        'zero shot': f'{model} (zero shot)' + (' ★' if bv == 'zero shot' else ''),
        'few shot':  f'{model} (few shot)'  + (' ★' if bv == 'few shot'  else ''),
        'persona':   f'{model} (persona)'   + (' ★' if bv == 'persona'   else ''),
        'PPI':       f'PPI [★ {bv}]',
        'PDI':       f'PDI [★ {bv}]',
    }

    for m in METHODS:
        line = f'{method_labels[m]:<{label_w}}'

        # Avg column = mean over subgroups of per-subgroup metric
        avg_cov = np.nanmean([coverage_pct(cov_d[m][:, i]) for i in sub_idx])
        avg_del = np.nanmean([delta_pp(ests_d[m][:, i], true_thetas[i]) for i in sub_idx])
        line += '  ' + fmt(avg_cov, cell_w, 1) + ' ' + fmt(avg_del, cell_w, 2)

        for i in sub_idx:
            cov = coverage_pct(cov_d[m][:, i])
            dlt = delta_pp(ests_d[m][:, i], true_thetas[i])
            line += '  ' + fmt(cov, cell_w, 1) + ' ' + fmt(dlt, cell_w, 2)
        print(line)

print(f'Cov. = coverage % (target {ci_level}%) | Δ = mean |est - true theta| in percentage points')
print(f'★ marks the prompting variant used for PPI/PDI. n_human={n_human}, n_trials={n_trials}, N={N_mm}')
print()

for model in all_models:
    if model not in results:
        continue
    r = results[model]
    print('=' * 100)
    print(f'MODEL: {model}   (best prompting: {r["best_variant"]})')
    print('=' * 100)
    for dim_name, subgroups in dimensions.items():
        print(f'\n[{dim_name}]')
        print_dim_table(model, dim_name, subgroups, r['ests'], r['covered'], true_thetas)
    print()

In [ ]:
# Mean human-sample allocation per subgroup (PDI adaptive sampling), averaged across trials.
print(f'Mean human samples allocated per subgroup (n_human={n_human}, {n_trials} trials)')
print()
header = f'{"Model":<22} {"Variant":<10} ' + ' '.join(f'{g:>14}' for g in all_groups)
print(header)
print('-' * len(header))
for model in all_models:
    if model not in results:
        continue
    r = results[model]
    means = r['samples_per_g'].mean(axis=0)
    row = f'{model:<22} {r["best_variant"]:<10} ' + ' '.join(f'{m:>14.1f}' for m in means)
    print(row)